In [3]:
import os
import glob
import shutil

# 1. 目錄設定
raw_pdb_dir = os.path.join("MCGLPPI_RawData", "pdbs", "m2_pdbbind_dimer_strict")
pdbqt_dir = os.path.join("ProAffinity-GNN", "data", "pdbqt")
mole2_dir = os.path.join("ProAffinity-GNN", "data", "FASTA", "2mole")
mixed_dir = os.path.join("ProAffinity-GNN", "data", "FASTA", "mixed")
chain_index_path = os.path.join("ProAffinity-GNN", "data", "chain_index.txt")

# 清空舊資料夾
for d in [mole2_dir, mixed_dir]:
    if os.path.exists(d): shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

aminoacid_abbr = {
    'GLY': 'G', 'ALA': 'A', 'VAL': 'V', 'LEU': 'L', 'ILE': 'I', 
    'PHE': 'F', 'TRP': 'W', 'TYR': 'Y', 'ASP': 'D', 'ASN': 'N', 
    'GLU': 'E', 'LYS': 'K', 'GLN': 'Q', 'MET': 'M', 'SER': 'S', 
    'THR': 'T', 'CYS': 'C', 'PRO': 'P', 'HIS': 'H', 'ARG': 'R',
    'HID': 'H', 'HIE': 'H', 'HIP': 'H', 'CYX': 'C', 'ASH': 'D', 'GLH': 'E'
}

def get_itp_length(itp_path):
    length = 0
    in_atoms = False
    last_resnr = ""
    if not os.path.exists(itp_path): return 0
    
    with open(itp_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith(';'): continue
            if line.startswith('[ atoms ]'):
                in_atoms = True
                continue
            if line.startswith('[') and in_atoms: break
            if in_atoms:
                parts = line.split()
                if len(parts) >= 4:
                    resnr = parts[2]
                    if resnr != last_resnr: # 避免同一個氨基酸的多個珠子重複算
                        length += 1
                        last_resnr = resnr
    return length

pdbqt_files = glob.glob(os.path.join(pdbqt_dir, "*_atom_processed.pdbqt"))
success = 0
fallback_count = 0

with open(chain_index_path, 'w') as index_file:
    for pdbqt_file in pdbqt_files:
        pdb_id = os.path.basename(pdbqt_file).split('_')[0].lower()
        
        # [步驟 A]：讀取 PDBQT，依照 Chain ID 整理出存活的序列
        chains_seq = {}
        last_res_num = {}
        chain_order = [] # 紀錄鍊的出現順序
        
        with open(pdbqt_file, 'r') as f:
            for line in f:
                if line.startswith("ATOM") or line.startswith("HETATM"):
                    if line[12:16].strip() == 'CA':
                        cid = line[21]
                        res_type = line[17:21].strip()
                        res_num = line[22:27].strip()
                        
                        if cid not in chains_seq:
                            chains_seq[cid] = ""
                            last_res_num[cid] = ""
                            chain_order.append(cid)
                            
                        if res_num != last_res_num[cid]:
                            chains_seq[cid] += aminoacid_abbr.get(res_type, 'X')
                            last_res_num[cid] = res_num
                            
        if len(chain_order) < 2:
            continue # 如果被 OpenBabel 刪到連兩條鍊都不剩，那就真的沒救了
            
        # [步驟 B]：利用 _A.itp 的長度來精準分組
        itp_a_path = os.path.join(raw_pdb_dir, pdb_id, f"{pdb_id}-cg_A.itp")
        len_a_itp = get_itp_length(itp_a_path)
        
        best_split_idx = 1
        # 如果找得到 _A.itp，我們就依照長度找出最完美的切割點
        if len_a_itp > 0:
            min_diff = float('inf')
            current_len = 0
            for i in range(len(chain_order) - 1):
                current_len += len(chains_seq[chain_order[i]])
                diff = abs(current_len - len_a_itp)
                if diff < min_diff:
                    min_diff = diff
                    best_split_idx = i + 1
        else:
            # 如果連 _A.itp 都沒有 (那 142 個檔案)，啟動備案：將最後一條鍊視為 Ligand
            best_split_idx = len(chain_order) - 1
            fallback_count += 1
            
        # [步驟 C]：正式分裝成兩組
        group1_cids = chain_order[:best_split_idx]
        group2_cids = chain_order[best_split_idx:]
        
        seq1 = "".join([chains_seq[c] for c in group1_cids])
        seq2 = "".join([chains_seq[c] for c in group2_cids])
        
        # [步驟 D]：輸出檔案 (同時滿足 2mole 和 mixed)
        # 標頭格式：>1A22_1|A,B,C|Fake...
        header1 = f">{pdb_id.upper()}_1|{','.join(group1_cids)}|Fake Protein|Fake Species"
        header2 = f">{pdb_id.upper()}_2|{','.join(group2_cids)}|Fake Protein|Fake Species"
        
        for target_dir in [mole2_dir, mixed_dir]:
            with open(os.path.join(target_dir, f"{pdb_id.upper()}_1.fasta"), 'w') as f:
                f.write(f"{header1}\n{seq1}\n")
            with open(os.path.join(target_dir, f"{pdb_id.upper()}_2.fasta"), 'w') as f:
                f.write(f"{header2}\n{seq2}\n")
                
        # 寫入 Index：例如 ABC; D;
        index_str = f"{''.join(group1_cids)}; {''.join(group2_cids)};"
        index_file.write(f"{pdb_id}\t{index_str}\n")
        
        success += 1

print(f"\n✅ 大功告成！完美重建了 {success} 個 Graph 必須的 FASTA 結構！")
if fallback_count > 0:
    print(f"⚠️ 其中有 {fallback_count} 個 PDB 缺少 .itp 檔案，已自動啟動備案邏輯成功拯救。")


✅ 大功告成！完美重建了 1245 個 Graph 必須的 FASTA 結構！
